# Metasyn Tutorial: Time-Series Dependencies and Logical Constraints

In this tutorial, we explore how to model time-series data with start and end dates, and how to add column dependency relationships and constraints to synthetic data using the implemented dunder methods.

TODO:
- how to handle duration in combination with start/end dates when it is already present in the original dataset. For example Length_of_stay column in the dataset used in this tutorial.

### 0. Install and import

First, let's install metasyn if you haven't done so already.

In [1]:
# %pip install metasyn

Now let's import the packages we need.

In [2]:
import polars as pl

from metasyn import demo_data
from metasyn.builder import MetaFrameBuilder
from metasyn.distribution import DiscreteTruncatedNormalDistribution
from metasyn.distribution.base import (
    ColumnReference,
    IfThenElse,
)

### 1. Synthesising time-series data

A common challenge with time-series datasets is that columns can be related to each other. For example, in a hospital admissions dataset the `discharge_date` should always come *after* the `admission_date`. By default, metasyn treats each column independently, so this constraint is not preserved.

Let's load our hospital admissions dataset to see this in action.

In [3]:
df = pl.read_csv("../metasyn/demo/demo_hospital_2.csv")
df = df.with_columns(
    pl.col("Patient_id").cast(pl.Int64),
    pl.col("Admission_date").str.to_datetime(),
    pl.col("Discharge_date").str.to_datetime(),
    pl.col("Length_of_stay").str.extract(r"(\d+)").cast(pl.Int64),
    pl.col("Sex").cast(pl.Categorical),
)

df

Patient_id,Admission_date,Discharge_date,Length_of_stay,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat
1,2023-01-04 00:00:00,2023-01-06 00:00:00,2,44,164,67,"""F"""
2,2023-01-08 00:00:00,2023-01-18 00:00:00,10,78,154,57,"""F"""
3,2023-01-08 00:00:00,2023-01-19 00:00:00,11,18,167,77,"""M"""
4,2023-01-08 00:00:00,2023-01-24 00:00:00,16,83,177,95,"""F"""
5,2023-01-15 00:00:00,2023-01-21 00:00:00,6,9,138,25,"""M"""
…,…,…,…,…,…,…,…
96,2024-12-10 00:00:00,2024-12-21 00:00:00,11,81,173,71,"""M"""
97,2024-12-19 00:00:00,2024-12-25 00:00:00,6,21,180,70,"""M"""
98,2024-12-19 00:00:00,2025-01-04 00:00:00,16,9,152,30,"""M"""


If we fit and synthesise without any additional instructions, the dates are generated independently. Let's check how often this produces invalid results.

In [4]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

synth = builder.fit().synthesize()

n_invalid = (synth["Discharge_date"] < synth["Admission_date"]).sum()
print(f"\nRows where Discharge_date < Admission_date: {n_invalid} / {len(synth)}")

synth.head()

  Patient_id: 100%|██████████| 8/8 [00:00<00:00, 727.51variables/s]


Rows where Discharge_date < Admission_date: 53 / 100


Patient_id,Admission_date,Discharge_date,Length_of_stay,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat
1,2024-02-14 00:00:00,2023-02-03 00:00:00,18,69,147,6,"""M"""
2,2024-02-17 00:00:00,2023-10-27 00:00:00,5,21,186,51,"""F"""
3,2024-06-10 00:00:00,2023-05-05 00:00:00,6,77,158,57,"""M"""
4,2024-09-14 00:00:00,2024-06-16 00:00:00,1,41,204,58,"""M"""
5,2024-08-31 00:00:00,2023-07-07 00:00:00,3,38,139,49,"""F"""


As we can see, there are some invalid rows. To fix this, we can add a hidden `duration_days` column that holds the difference in days between `Discharge_date` and `Admission_date`. Metasyn fits a distribution to the new column, which can then be used to compute `Discharge_date = Admission_date + duration_days`. Setting `hidden=True` ensures the helper column does **not** appear in the final output.

Note that `ColumnReference` refers to the values in a column at synthesis time, and can be used to build expressions between columns.

In [5]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

# Add a column and mark as hidden
builder.add_column("duration_days", hidden=True)

# Create a series for the difference between discharge_date and admission_date, and use it to derive discharge_date
builder["duration_days"].series = ColumnReference("Discharge_date") - ColumnReference("Admission_date")
builder["Discharge_date"].distribution = ColumnReference("Admission_date") + ColumnReference("duration_days")

synth = builder.fit().synthesize()

n_invalid = (synth["Discharge_date"] < synth["Admission_date"]).sum()
print(f"\nRows where Discharge_date < Admission_date: {n_invalid} / {len(synth)}")

synth

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 983.76variables/s]


Rows where Discharge_date < Admission_date: 0 / 100


Patient_id,Admission_date,Discharge_date,Length_of_stay,Age,Height_cm,Weight_kg,Sex
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat
1,2023-06-06 00:00:00,2023-06-17 00:00:00,4,63,149,68,"""F"""
2,2024-01-14 00:00:00,2024-01-17 00:00:00,8,15,149,61,"""F"""
3,2024-10-26 00:00:00,2024-11-05 00:00:00,3,16,181,39,"""M"""
4,2024-10-24 00:00:00,2024-11-02 00:00:00,12,42,195,44,"""F"""
5,2024-12-07 00:00:00,2024-12-10 00:00:00,16,75,165,69,"""M"""
…,…,…,…,…,…,…,…
96,2024-04-25 00:00:00,2024-04-29 00:00:00,16,82,156,94,"""F"""
97,2024-03-13 00:00:00,2024-04-01 00:00:00,14,32,140,49,"""F"""
98,2023-06-09 00:00:00,2023-06-20 00:00:00,5,53,176,85,"""F"""


No more invalid rows.

### 2. Logical expressions and conditions

If a boolean column is fully determined by another column, you can encode that dependency directly. For example, if the dataset contains an `Adult` column, you can define it as `builder["Adult"].distribution = ColumnReference("Age") > 18`. This keeps synthesized data internally consistent without post-processing.


The comparison and logical operators let you derive boolean columns from expressions. The table below lists the implemented operators and example usage. For readability, the examples show column names directly, but in practice, reference a column with `ColumnReference()`, for example `ColumnReference("Age")`.

| Dunder | Operator | Example |
|---|---|---|
| `__add__` | `a + b` | `Age + 10` |
| `__sub__` | `a - b` | `Age - 10` |
| `__mul__` | `a * b` | `Weight_kg * 2` |
| `__truediv__` | `a / b` | `Weight_kg / 2` |
| `__pow__` | `a ** b` | `Weight_kg ** 2` |
| `__neg__` | `0 - a` | `0 - Weight_kg` |
| `__invert__` | `not a` | `not Adult` |
| `__and__` | `a & b` | `Adult & (Sex == "male")` |
| `__or__` | `a \| b` | `Adult \| Child` |
| `__eq__` | `a == b` | `Sex == "female"` |
| `__ne__` | `a != b` | `Sex != "male"` |
| `__lt__` | `a < b` | `Age < 18` |
| `__gt__` | `a > b` | `Age > 17` |
| `__le__` | `a <= b` | `Age <= 17` |
| `__ge__` | `a >= b` | `Age >= 18` |



In [ ]:
# Example without relations or constraints

# Create example data with boolean "Adult" column
df = df.with_columns(
    (pl.col("Age") > 18).alias("Adult"),
)

builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

synth = builder.fit().synthesize()

n_mismatches = synth.filter(
    (pl.col("Age") > 18) & ~pl.col("Adult")
).height

print(f"\nRows where age > 18 but Adult is false: {n_mismatches} / {len(synth)}")

synth[["Age", "Adult"]]

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 798.09variables/s]


Rows where age > 18 but Adult is false: 16 / 100


Age,Adult
i64,bool
34,false
22,true
9,true
84,true
77,false
…,…
85,false
37,true
7,true


Let's fix the inconsistency.

In [7]:
# Add relations or constraints
builder["Adult"].distribution = ColumnReference("Age") > 18

synth = builder.fit().synthesize()

n_mismatches = synth.filter(
    (pl.col("Age") > 18) & ~pl.col("Adult")
).height

print(f"\nRows where age > 18 but Adult is false: {n_mismatches} / {len(synth)}")

synth[["Age", "Adult"]]

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 794.61variables/s]


Rows where age > 18 but Adult is false: 0 / 100


Age,Adult
i64,bool
11,false
58,true
81,true
46,true
54,true
…,…
14,false
64,true
28,true


#### IfThenElse example

Sometimes the distribution of a column depends on the value of another column. For example, `Height_cm` tends to differ between male and female patients. With `IfThenElse`, you can specify a different distribution or value for each group.

In [8]:
# Males: TruncatedNormal centred at 180 cm; females: centred at 170 cm
builder["Height_cm"].distribution = IfThenElse(
    ColumnReference("Sex") == "M",
    DiscreteTruncatedNormalDistribution(lower=160, upper=200, mean=180, sd=10),
    DiscreteTruncatedNormalDistribution(lower=150, upper=190, mean=170, sd=10),
)

synth = builder.fit().synthesize()
synth[["Sex", "Height_cm"]].head(10)

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 730.40variables/s]


Sex,Height_cm
cat,i64
"""F""",166
"""M""",188
"""F""",181
"""M""",186
"""F""",162
"""M""",178
"""F""",163
"""M""",192
"""M""",167


Actually, this dataset includes more complex dependencies: `Height_cm` and `Weight_kg` are strongly associated with age.
For example, a 6-year-old is very unlikely to be 200 cm tall or weigh 100 kg.

To make synthetic data more realistic, we can divide age into categories and assign plausible ranges to each group:

- **0-2 years:** height 50-90 cm, weight 3-14 kg
- **3-12 years:** height 95-155 cm, weight 14-50 kg
- **13-17 years:** height 145-190 cm, weight 40-90 kg
- **18+ years:** height 145-210 cm, weight 45-150 kg

These ranges are useful defaults for synthetic data generation, but edge-case combinations can still appear.

In the example below, we combine logical and conditional operators to define distributions across `Age` ranges.

In [13]:
builder = MetaFrameBuilder()
builder.add_dataframe(df, None)

builder["Weight_kg"].distribution = IfThenElse(
    ColumnReference("Age") <= 2,
    DiscreteTruncatedNormalDistribution(lower=3, upper=14, mean=8, sd=2),
    IfThenElse(
        ColumnReference("Age") <= 12,
        DiscreteTruncatedNormalDistribution(lower=14, upper=50, mean=30, sd=10),
        IfThenElse(
            ColumnReference("Age") <= 17,
            DiscreteTruncatedNormalDistribution(lower=40, upper=90, mean=65, sd=10),
            DiscreteTruncatedNormalDistribution(lower=50, upper=120, mean=75, sd=15),
        ),
    ),
)

builder["Height_cm"].distribution = IfThenElse(
    ColumnReference("Age") <= 2,
    DiscreteTruncatedNormalDistribution(lower=50, upper=90, mean=70, sd=10),
    IfThenElse(
        ColumnReference("Age") <= 12,
        DiscreteTruncatedNormalDistribution(lower=95, upper=155, mean=125, sd=15),
        IfThenElse(
            ColumnReference("Age") <= 17,
            DiscreteTruncatedNormalDistribution(lower=145, upper=190, mean=167, sd=12),
            DiscreteTruncatedNormalDistribution(lower=145, upper=210, mean=172, sd=12),
        ),
    ),
)

synth = builder.fit().synthesize()

synth

  Patient_id: 100%|██████████| 9/9 [00:00<00:00, 591.78variables/s]


Patient_id,Admission_date,Discharge_date,Length_of_stay,Age,Height_cm,Weight_kg,Sex,Adult
i64,datetime[μs],datetime[μs],i64,i64,i64,i64,cat,bool
1,2024-06-17 00:00:00,2024-08-05 00:00:00,9,3,118,31,"""M""",true
2,2024-12-15 00:00:00,2023-05-20 00:00:00,11,53,172,74,"""M""",true
3,2023-06-19 00:00:00,2024-03-22 00:00:00,10,50,171,77,"""F""",false
4,2023-08-29 00:00:00,2025-01-03 00:00:00,13,7,125,30,"""F""",true
5,2024-01-11 00:00:00,2023-03-11 00:00:00,15,78,174,92,"""F""",true
…,…,…,…,…,…,…,…,…
96,2023-09-21 00:00:00,2024-10-06 00:00:00,8,38,184,78,"""F""",true
97,2024-08-06 00:00:00,2024-11-03 00:00:00,2,51,157,78,"""F""",true
98,2023-04-22 00:00:00,2023-10-30 00:00:00,13,68,151,89,"""F""",false


As you can see, height and weight are adjusted to match the different age groups in the synthesized data.